# Nucleotide Transformer v2 test

In [1]:
import pandas as pd
import os

## Load data

In [3]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


## Get embedding using JAX

In [ ]:
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

/home/carrillo/micromamba/envs/pbi-nt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Select a model from '500M_human_ref', '500M_1000G', '2B5_1000G', '2B5_multi_species', '50M_multi_species_v2', '100M_multi_species_v2', '250M_multi_species_v2', '500M_multi_species_v2', '1B_agro_nt'
model_name = '50M_multi_species_v2'

In [ ]:
max_length = int(max(map(len, phages_df["phage_sequence"][:2]))/6) + 1
max_length

12414

In [ ]:
# Get pretrained model
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name=model_name,
    embeddings_layers_to_save=(12,),
    attention_maps_to_save=((1, 4), (7, 16)),
    max_positions=32,
)
forward_fn = hk.transform(forward_fn)

Downloaded model's hyperparameters.
Downloaded model's weights...


In [ ]:
# Get data and tokenize it
sequences = [
    "ATTCCGAAATCGCTGACCGATCGTACGAAA",
    "ATTTCTCTCTCTCTCTGAGATCGATCGATCGATATCTCTCGAGCTAGC",
]
# sequences = list(phages_df["phage_sequence"])[:2]
tokens_ids = [b[1] for b in tokenizer.batch_tokenize(sequences)]
tokens_str = [b[0] for b in tokenizer.batch_tokenize(sequences)]
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [ ]:
# Initialize random key
random_key = jax.random.PRNGKey(0)

# Infer
outs = forward_fn.apply(parameters, random_key, tokens)
print(outs.keys())

dict_keys(['attention_map_layer_1_number_4', 'attention_map_layer_7_number_16', 'embeddings_12', 'logits'])


In [ ]:
embeddings = outs["embeddings_12"][:, 1:, :]  # removing CLS token
padding_mask = jnp.expand_dims(tokens[:, 1:] != tokenizer.pad_token_id, axis=-1)
masked_embeddings = embeddings * padding_mask  # multiply by 0 pad tokens embeddings
sequences_lengths = jnp.sum(padding_mask, axis=1)
mean_embeddings = jnp.sum(masked_embeddings, axis=1) / sequences_lengths
print(mean_embeddings.shape)

(2, 512)


In [ ]:
mean_embeddings

Array([[ 0.0307321 ,  0.28487912, -0.0241097 , ..., -0.00697614,
        -0.10268458, -0.20914476],
       [-0.17261767, -0.13952518, -0.12027154, ...,  0.15793578,
         0.15447846, -0.12072094]], dtype=float32)

## Embeddings using 🤗 transformers

In [4]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

In [5]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device = "cpu"
device

'cpu'

In [6]:
# Import the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)

In [7]:
# Choose the length to which the input sequences are padded. By default, the 
# model max length is chosen, but feel free to decrease it as the time taken to 
# obtain the embeddings increases significantly with it.
max_length = tokenizer.model_max_length
max_length

2048

In [8]:
# Create a dummy dna sequence and tokenize it
# sequences = ["ATTCCGATTCCGATTCCG", "ATTTCTCTCTCTCTCTGAGATCGATCGATCGAT"]
# sequences = phages_df["phage_sequence"].iloc[:1]
sequences = bacteria_df["bacterium_sequence"].iloc[:1]
print(f"{len(sequences)}, {list(map(len, sequences))}")
tokens_ids = tokenizer.batch_encode_plus(sequences, return_tensors="pt")["input_ids"].to(device)
tokens_ids.shape

1, [6988208]


Token indices sequence length is longer than the specified maximum sequence length for this model (1164704 > 2048). Running this sequence through the model will result in indexing errors


torch.Size([1, 1164704])

In [15]:
# Split the sequence in chunks of max_length
tokens_list = []
for token_ids in tokens_ids:
    for i in range(0, len(token_ids), max_length):
        # Pad with zeros if the last chunk is smaller than max_length
        l = np.zeros(max_length, dtype=int)
        l[: min(max_length, len(token_ids) - i)] = token_ids[i : i + max_length]
        tokens_list.append(l)
tokens_ids = torch.tensor(tokens_list, device=device)
tokens_ids.shape

old_tokens = tokens_ids.clone()
old_tokens.shape

torch.Size([569, 2048])

In [20]:
tokens_ids = old_tokens[:20]
tokens_ids.shape

torch.Size([20, 2048])

In [106]:
# tokens_ids = tokens_ids[0][:max_length].unsqueeze(0)
tokens_ids[0][3*max_length:] = 0
print(tokenizer.SPECIAL_TOKENS_ATTRIBUTES)
print(f"Pad token id: {tokenizer.pad_token_id}")
print(f"CLS token id: {tokenizer.cls_token_id}")
print(f"SEP token id: {tokenizer.sep_token_id}")
print(f"EOS token id: {tokenizer.eos_token_id}")
print(f"UNK token id: {tokenizer.unk_token_id}")
print(f"BOS token id: {tokenizer.bos_token_id}")


['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']
Pad token id: 1
CLS token id: 3
SEP token id: None
EOS token id: None
UNK token id: 0
BOS token id: None


In [21]:
model.to(device)

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(4107, 512, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(2050, 512, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-11): 12 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((512,), eps=1e-12, elementwi

In [23]:
# Compute the embeddings
attention_mask = tokens_ids != tokenizer.pad_token_id
torch_outs = model(
    tokens_ids,
    attention_mask=attention_mask,
    encoder_attention_mask=attention_mask,
    output_hidden_states=True
)

In [24]:
# Compute sequences embeddings
embeddings = torch_outs['hidden_states'][-1].detach().cpu().numpy()
print(f"Embeddings shape: {embeddings.shape}")
# print(f"Embeddings per token: {embeddings}")

# Add embed dimension axis
attention_mask_unsq = torch.unsqueeze(attention_mask, dim=-1).cpu()

# Compute mean embeddings per sequence
mean_sequence_embeddings = torch.sum(attention_mask_unsq*embeddings, axis=-2)/torch.sum(attention_mask_unsq, axis=1) # type: ignore
print(f"Mean sequence embeddings shape: {mean_sequence_embeddings.shape}")
print(f"Mean sequence embeddings: {mean_sequence_embeddings}")

Embeddings shape: (20, 2048, 512)
Mean sequence embeddings shape: torch.Size([20, 512])
Mean sequence embeddings: tensor([[-0.2122,  0.5409,  0.3378,  ..., -0.0918, -0.2062, -0.1044],
        [-0.1971,  0.5477,  0.4667,  ...,  0.1283, -0.3627, -0.1174],
        [-0.3607,  0.4304,  0.4759,  ...,  0.2467, -0.4095, -0.0058],
        ...,
        [-0.1575,  0.6821,  0.5164,  ...,  0.0983, -0.3472, -0.1387],
        [-0.3073,  0.6150,  0.4184,  ..., -0.0108, -0.3489, -0.1118],
        [-0.2330,  0.6369,  0.5058,  ...,  0.1466, -0.3411, -0.0150]])


In [88]:
zeroed = mean_sequence_embeddings
zeroed

tensor([[-3.8023e-01, -1.4281e-01,  2.2012e-01,  9.4747e-02,  3.6082e-01,
         -1.4568e-01, -5.2214e-01,  3.8816e-01, -3.5456e-01,  3.2087e-02,
          4.3009e-03, -1.1695e-01,  1.0030e-01, -3.5188e-01, -2.2118e-01,
          2.5403e-01,  3.4589e-01,  4.4435e-01,  3.9248e-01,  2.9103e-01,
          7.0244e-01,  1.9384e-01,  2.4440e-01, -4.1436e-01,  1.4583e-01,
         -6.5368e-01,  3.0931e-01,  1.4068e-01, -1.3681e-01, -1.6896e-01,
         -2.5449e-01,  4.1448e-02,  3.6799e-01,  1.7431e-01,  2.6486e-01,
          1.2319e-01, -3.2893e-01,  4.9520e-03,  1.7066e-01, -2.9404e-03,
         -5.2632e-01, -7.5079e-02, -2.6196e-01,  1.6313e-02, -9.8853e-01,
         -2.7598e-01, -2.2486e-01,  4.8689e-01, -2.7879e-01,  7.5270e-02,
          6.7399e-02,  3.7775e-01, -1.9904e-01, -3.0300e-02,  2.6878e-01,
         -2.3290e-01, -5.0733e-02, -1.0999e-01, -4.7226e-02,  5.7977e-01,
          6.6962e-02,  1.3214e-01, -2.2616e-01, -8.9857e-02, -2.4897e-01,
         -1.6742e-01, -3.2693e-02,  1.

## Finetuning NT2